# RGCA Full Research Pipeline for Kaggle

Run this notebook from top to bottom. It is designed to be self-contained and research-safe for the current baseline milestone.

It performs:

- Repository setup
- MIMIC pilot subset detection or preparation
- Dataset contract validation
- Controlled stress baseline experiments
- Pilot JPG hydration from PhysioNet, not the full 4.7 TB archive
- BioMedCLIP retrieval-only validation
- Evidence packaging for download

Expected final artifact:

```text
/kaggle/working/rgca_research_evidence_v0.zip
```


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import textwrap

print('Python:', sys.version)
print('Kaggle working exists:', Path('/kaggle/working').exists())

# -----------------------------
# User-facing configuration
# -----------------------------
KAGGLE_USERNAME = 'emmapi'
REPO_URL = 'https://github.com/pidoxy/RGCA.git'
PROJECT_ROOT = Path('/kaggle/working/RGCA')
WORK_DIR = Path('/kaggle/working/physionet')
STRESS_OUTPUT_DIR = Path('/kaggle/working/rgca_experiments/mimic_pilot_baseline_v0')
HYDRATED_SUBSET_JSONL = Path('/kaggle/working/rgca_hydrated_subset/mimic_subset.jsonl')
BIOMEDCLIP_OUTPUT_DIR = Path('/kaggle/working/rgca_experiments/biomedclip_retrieval_validation_v0')
ALL_EVIDENCE_ZIP = Path('/kaggle/working/rgca_research_evidence_v0.zip')
PRIVATE_DATASET_SLUG = 'rgca-private-dataset'
RUN_STRESS_BASELINE = True
RUN_BIOMEDCLIP_RETRIEVAL = True
RETRIEVAL_LIMIT = 400
EVAL_LIMIT = 100
BIOMEDCLIP_EVAL_LIMIT = 20
TOP_K = 3

# -----------------------------
# Helpers
# -----------------------------
def run(command, cwd=None, env=None):
    cwd = Path(cwd or PROJECT_ROOT)
    printable = []
    for part in command:
        text = str(part)
        if 'PHYSIONET' in text or len(text) > 120:
            printable.append(text[:120] + '...')
        else:
            printable.append(text)
    print('+', ' '.join(printable))
    return subprocess.run([str(part) for part in command], cwd=str(cwd), env=env, check=True)


def read_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def find_first(root, pattern):
    root = Path(root)
    if not root.exists():
        return None
    matches = sorted(root.rglob(pattern))
    return matches[0] if matches else None


def find_subset_jsonl():
    candidates = []
    for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if root.exists():
            candidates.extend(root.rglob('mimic_subset.jsonl'))
    candidates = sorted(set(candidates))
    print('Subset candidates:')
    for path in candidates[:20]:
        print(' -', path)
    return candidates[0] if candidates else None


def ensure_repo():
    if PROJECT_ROOT.exists():
        print('Repo already exists:', PROJECT_ROOT)
        try:
            run(['git', 'pull'], cwd=PROJECT_ROOT)
        except Exception as exc:
            print('Repo pull failed; continuing with existing checkout:', exc)
    else:
        run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], cwd=Path('/kaggle/working'))
    run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=PROJECT_ROOT)
    if str(PROJECT_ROOT / 'src') not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT / 'src'))
    print('Repo ready:', PROJECT_ROOT)


def prepare_subset_from_physionet(username, password):
    if not username or not password:
        raise RuntimeError('No attached mimic_subset.jsonl found, and PhysioNet secrets are missing.')
    env = os.environ.copy()
    env['PHYSIONET_PASS'] = password
    output_dir = Path('/kaggle/working/rgca_pilot_500')
    run([
        sys.executable,
        'scripts/kaggle_prepare_mimic_subset.py',
        '--physionet-user', username,
        '--work-dir', WORK_DIR,
        '--output-dir', output_dir,
        '--retrieval-limit', RETRIEVAL_LIMIT,
        '--eval-limit', EVAL_LIMIT,
    ], cwd=PROJECT_ROOT, env=env)
    subset = output_dir / 'data' / 'mimic_subset.jsonl'
    if not subset.exists():
        raise FileNotFoundError(f'Expected prepared subset missing: {subset}')
    return subset


def hydrate_pilot_images(subset_path, username, password):
    if not username or not password:
        raise RuntimeError('BioMedCLIP retrieval needs PHYSIONET_USERNAME and PHYSIONET_PASS Kaggle secrets.')

    # Prefer repo script when present, otherwise use inline fallback. This keeps the notebook runnable even
    # before the latest repo commit is available on Kaggle.
    script = PROJECT_ROOT / 'scripts' / 'kaggle_hydrate_mimic_images.py'
    if script.exists():
        env = os.environ.copy()
        env['PHYSIONET_PASS'] = password
        run([
            sys.executable,
            script,
            '--subset', subset_path,
            '--physionet-user', username,
            '--output-root', '/kaggle/working/physionet/mimic-cxr-jpg/files',
            '--updated-subset', HYDRATED_SUBSET_JSONL,
        ], cwd=PROJECT_ROOT, env=env)
        return HYDRATED_SUBSET_JSONL

    print('Hydration script not found; using inline fallback.')
    records = [json.loads(line) for line in Path(subset_path).read_text().splitlines() if line.strip()]
    out_root = Path('/kaggle/working/physionet/mimic-cxr-jpg/files')
    HYDRATED_SUBSET_JSONL.parent.mkdir(parents=True, exist_ok=True)
    netrc = Path.home() / '.netrc'
    netrc.write_text(f'machine physionet.org login {username} password {password}\n')
    netrc.chmod(0o600)
    downloaded = 0
    already_exists = 0
    updated = []
    base_url = 'https://physionet.org/files/mimic-cxr-jpg/2.1.0/files'
    try:
        for idx, row in enumerate(records, start=1):
            subject_id = str(row['subject_id'])
            study_id = str(row['study_id'])
            dicom_id = str(row['dicom_id'])
            rel = Path(f'p{subject_id[:2]}') / f'p{subject_id}' / f's{study_id}' / f'{dicom_id}.jpg'
            dest = out_root / rel
            if dest.exists() and dest.stat().st_size > 0:
                already_exists += 1
            else:
                dest.parent.mkdir(parents=True, exist_ok=True)
                url = f'{base_url}/{rel.as_posix()}'
                result = subprocess.run(['wget', '-q', '-c', '--netrc', url, '-O', str(dest)])
                if result.returncode != 0 or not dest.exists() or dest.stat().st_size == 0:
                    raise RuntimeError(f'Failed to download {url} -> {dest}')
                downloaded += 1
            row = dict(row)
            row['image_path'] = str(dest)
            updated.append(row)
            if idx % 25 == 0 or idx == len(records):
                print(f'hydrated {idx}/{len(records)} images; downloaded={downloaded}; already_exists={already_exists}')
    finally:
        netrc.unlink(missing_ok=True)

    HYDRATED_SUBSET_JSONL.write_text('\n'.join(json.dumps(row, ensure_ascii=True) for row in updated) + '\n')
    manifest = {
        'source_subset': str(subset_path),
        'hydrated_subset': str(HYDRATED_SUBSET_JSONL),
        'records': len(updated),
        'downloaded': downloaded,
        'already_exists': already_exists,
        'output_root': str(out_root),
    }
    (HYDRATED_SUBSET_JSONL.parent / 'image_hydration_manifest.json').write_text(json.dumps(manifest, indent=2))
    return HYDRATED_SUBSET_JSONL


def validate_subset(subset_path):
    from rgca_baseline.integrity import jsonl_fingerprint, validate_image_paths, validate_study_records
    from rgca_baseline.pipeline import load_studies
    studies = load_studies(subset_path)
    validation = validate_study_records(studies)
    image_validation = validate_image_paths(studies)
    summary = {
        'subset': str(subset_path),
        'fingerprint': jsonl_fingerprint(subset_path),
        'dataset_validation': validation,
        'image_validation': image_validation,
    }
    print(json.dumps(summary, indent=2)[:5000])
    if not validation['valid']:
        raise RuntimeError('Dataset contract validation failed.')
    return summary


def zip_dir(source_dir, zip_path):
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', str(source_dir))
    print('zip:', zip_path, 'exists=', zip_path.exists(), 'size_mb=', round(zip_path.stat().st_size / (1024 * 1024), 3))

# -----------------------------
# Execute full pipeline
# -----------------------------
ensure_repo()
physionet_username = read_secret('PHYSIONET_USERNAME') or read_secret('PHYSIONET_USER')
physionet_password = read_secret('PHYSIONET_PASS') or read_secret('PHYSIONET_PASSWORD')
print('PHYSIONET_USERNAME set:', bool(physionet_username))
print('PHYSIONET_PASS set:', bool(physionet_password))

subset = find_subset_jsonl()
if subset is None:
    subset = prepare_subset_from_physionet(physionet_username, physionet_password)
print('Using subset:', subset)
validate_subset(subset)

if RUN_STRESS_BASELINE:
    run([
        sys.executable,
        'scripts/kaggle_bootstrap_baseline.py',
        '--subset-jsonl', subset,
        '--execution-mode', 'stress',
        '--output-dir', STRESS_OUTPUT_DIR,
        '--pilot-output-dir', Path(subset).parent.parent if Path(subset).parent.name == 'data' else Path(subset).parent,
        '--overwrite',
    ], cwd=PROJECT_ROOT)

if RUN_BIOMEDCLIP_RETRIEVAL:
    hydrated_subset = hydrate_pilot_images(subset, physionet_username, physionet_password)
    validate_subset(hydrated_subset)
    run([sys.executable, '-m', 'pip', 'install', '-q', 'open_clip_torch', 'pillow'], cwd=PROJECT_ROOT)
    if BIOMEDCLIP_OUTPUT_DIR.exists():
        shutil.rmtree(BIOMEDCLIP_OUTPUT_DIR)
    run([
        sys.executable,
        'scripts/run_retrieval_validation.py',
        '--subset', hydrated_subset,
        '--backend', 'biomedclip',
        '--output-dir', BIOMEDCLIP_OUTPUT_DIR,
        '--top-k', TOP_K,
        '--eval-limit', BIOMEDCLIP_EVAL_LIMIT,
    ], cwd=PROJECT_ROOT)

zip_dir(Path('/kaggle/working/rgca_experiments'), ALL_EVIDENCE_ZIP)

# Compact final report.
final = {
    'stress_summary': str(STRESS_OUTPUT_DIR / 'bootstrap_summary.json'),
    'biomedclip_summary': str(BIOMEDCLIP_OUTPUT_DIR / 'retrieval_validation_summary.json'),
    'hydrated_subset': str(HYDRATED_SUBSET_JSONL),
    'evidence_zip': str(ALL_EVIDENCE_ZIP),
}
for name, path in final.items():
    p = Path(path)
    print(name, path, 'exists=', p.exists(), 'size_mb=', round(p.stat().st_size/(1024*1024), 3) if p.exists() else None)

if (BIOMEDCLIP_OUTPUT_DIR / 'retrieval_validation_summary.json').exists():
    s = json.loads((BIOMEDCLIP_OUTPUT_DIR / 'retrieval_validation_summary.json').read_text())
    print('BIOMEDCLIP compact:', json.dumps({
        'backend': s.get('backend'),
        'top_k': s.get('top_k'),
        'retrieval_pool_size': s.get('retrieval_pool_size'),
        'eval_size': s.get('eval_size'),
        'image_validation': s.get('image_validation'),
        'outputs': s.get('outputs'),
    }, indent=2))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Kaggle working exists: True
+ git clone https://github.com/pidoxy/RGCA.git /kaggle/working/RGCA


Cloning into '/kaggle/working/RGCA'...


+ /usr/bin/python3 -m pip install -q -e .
Repo ready: /kaggle/working/RGCA
PHYSIONET_USERNAME set: True
PHYSIONET_PASS set: True
Subset candidates:
 - /kaggle/input/datasets/emmapi/rgca-private-dataset/mimic_subset.jsonl
Using subset: /kaggle/input/datasets/emmapi/rgca-private-dataset/mimic_subset.jsonl
{
  "subset": "/kaggle/input/datasets/emmapi/rgca-private-dataset/mimic_subset.jsonl",
  "fingerprint": {
    "path": "/kaggle/input/datasets/emmapi/rgca-private-dataset/mimic_subset.jsonl",
    "sha256": "89187a1df84a81506e12762505722a703f4ae916cc05cdb7c86c91d2ef69f3b1",
    "rows": 500
  },
  "dataset_validation": {
    "total": 500,
    "retrieval_pool": 400,
    "eval": 100,
    "missing_required_count": 0,
    "missing_required_study_ids": [],
    "duplicate_study_ids": [],
    "valid": true
  },
  "image_validation": {
    "total": 500,
    "missing_image_count": 500,
    "missing_image_examples": [
      "/kaggle/working/physionet/mimic-cxr-jpg/files/p10/p10000032/s50414267/02aa8

{
  "created_at_utc": "2026-08-02T15:39:57.923505+00:00",
  "subset": "/kaggle/working/rgca_hydrated_subset/mimic_subset.jsonl",
  "dataset_fingerprint": {
    "path": "/kaggle/working/rgca_hydrated_subset/mimic_subset.jsonl",
    "sha256": "89187a1df84a81506e12762505722a703f4ae916cc05cdb7c86c91d2ef69f3b1",
    "rows": 500
  },
  "dataset_validation": {
    "total": 500,
    "retrieval_pool": 400,
    "eval": 100,
    "missing_required_count": 0,
    "missing_required_study_ids": [],
    "duplicate_study_ids": [],
    "valid": true
  },
  "image_validation": {
    "total": 500,
    "missing_image_count": 0,
    "missing_image_examples": [],
    "valid": true
  },
  "backend": "biomedclip",
  "retrieval_plan": {
    "backend": "biomedclip",
    "similarity_space": "shared image-text embedding space",
    "notes": "Recommended first real retrieval path. Query with target image embedding and retrieve nearest studies, then pass their reports to the generator."
  },
  "top_k": 3,
  "retriev

## Inspect Evidence Files

Run this after the main pipeline if you want a readable inventory of the generated outputs.


In [2]:
from pathlib import Path
import json

for path in [
    Path('/kaggle/working/rgca_research_evidence_v0.zip'),
    Path('/kaggle/working/rgca_hydrated_subset/mimic_subset.jsonl'),
    Path('/kaggle/working/rgca_hydrated_subset/image_hydration_manifest.json'),
    Path('/kaggle/working/rgca_experiments/mimic_pilot_baseline_v0/bootstrap_summary.json'),
    Path('/kaggle/working/rgca_experiments/biomedclip_retrieval_validation_v0/retrieval_validation_summary.json'),
]:
    print(path, 'exists=', path.exists(), 'size_mb=', round(path.stat().st_size/(1024*1024), 3) if path.exists() else None)

summary = Path('/kaggle/working/rgca_experiments/biomedclip_retrieval_validation_v0/retrieval_validation_summary.json')
if summary.exists():
    data = json.loads(summary.read_text())
    print(json.dumps(data, indent=2)[:5000])


/kaggle/working/rgca_research_evidence_v0.zip exists= True size_mb= 0.748
/kaggle/working/rgca_hydrated_subset/mimic_subset.jsonl exists= True size_mb= 0.818
/kaggle/working/rgca_hydrated_subset/image_hydration_manifest.json exists= True size_mb= 0.0
/kaggle/working/rgca_experiments/mimic_pilot_baseline_v0/bootstrap_summary.json exists= True size_mb= 0.001
/kaggle/working/rgca_experiments/biomedclip_retrieval_validation_v0/retrieval_validation_summary.json exists= True size_mb= 0.001
{
  "created_at_utc": "2026-08-02T15:39:57.923505+00:00",
  "subset": "/kaggle/working/rgca_hydrated_subset/mimic_subset.jsonl",
  "dataset_fingerprint": {
    "path": "/kaggle/working/rgca_hydrated_subset/mimic_subset.jsonl",
    "sha256": "89187a1df84a81506e12762505722a703f4ae916cc05cdb7c86c91d2ef69f3b1",
    "rows": 500
  },
  "dataset_validation": {
    "total": 500,
    "retrieval_pool": 400,
    "eval": 100,
    "missing_required_count": 0,
    "missing_required_study_ids": [],
    "duplicate_study_i